# Projet: Identification de pneumonie  à partir d'un réseau de neurones

On dispose d'un jeu de données de scanners de poumons.
L'objectif ici est de créer un modèle de qui identifie correctement si un poumon est atteint (0)ou non (1) de pneumonie à partir d'un réseau de neurones convolutionnels from scratch et d'exposer ce modèle à l'aide d'une API(FastAPI).

# Chargement et préparation des données

**Librairies**

In [4]:
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms, datasets, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset, random_split
import multipart

Les images sont ensuite transformées en format 224 x 224 puis en tenseur.

In [5]:
transform = transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor()])

In [6]:
data = ImageFolder('/home/paul/chest_xray/All', transform= transform)

Les noms des classes sont sauvegardés afin de les réutiliser dans l'API.

In [12]:
classes_name = data.classes
joblib.dump(classes_name,'classes_name.joblib')

['classes_name.joblib']

**On sépare ensuite les images d'apprentissage et les images de test**

In [68]:
train_size = int(0.8*len(data))
test_size = len(data) - train_size
trainset, testset = random_split(data, [train_size,test_size])

**On charge les images en lot de 64 afin d'accelérer les calculs.**

In [69]:
train_loader = DataLoader(trainset, batch_size= 64, shuffle= True)
test_loader = DataLoader(testset, batch_size= 64, shuffle= True)

**Ensuite, il faut calculer la moyenne et l'ecart-type approximatif de chaque couche de chaque image afin de normaliser les images.**

In [70]:
moyenne = torch.zeros(3)
ecart_type = torch.zeros(3)
n_batch = 0
for image, _ in train_loader:
    moyenne += image.mean(dim=[0,2,3])
    ecart_type += image.std(dim=[0,2,3])
    n_batch += 1
moyenne = moyenne / n_batch
ecart_type = ecart_type / n_batch

In [71]:
print(f'Moyenne par canal: {moyenne} | Ecart-type par canal: {ecart_type}')

Moyenne par canal: tensor([0.4827, 0.4827, 0.4827]) | Ecart-type par canal: tensor([0.2359, 0.2359, 0.2359])


**On peut enfin normaliser et augmenter l'échantillon d'images**

In [72]:
transform = transforms.Compose([transforms.Resize((224,224)), transforms.AugMix(),transforms.ToTensor(),transforms.Normalize(mean=moyenne, std = ecart_type)])
data = ImageFolder('/home/paul/chest_xray/All', transform= transform)
trainset, testset = random_split(data, [train_size,test_size])
train_loader = DataLoader(trainset, batch_size= 64, shuffle= True)
test_loader = DataLoader(testset, batch_size= 64, shuffle= True)

## Entrainement à l'aide de réseau de neurones convolutionel from scratch

In [73]:
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels= 3, out_channels= 16, kernel_size= 3, padding= 1, stride= 1) #(16,32,224,224)
        self.pool1 = nn.MaxPool2d(2,2)
        self.batch_norm1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(in_channels= 16, out_channels= 32, kernel_size= 3, padding= 1, stride= 1) #(16,64,112,112)
        self.pool2 = nn.MaxPool2d(2,2)
        self.batch_norm2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(in_channels= 32, out_channels= 64, kernel_size= 3, padding= 1, stride= 1) #(16,128,56,56)
        self.pool3 = nn.MaxPool2d(2,2)
        self.batch_norm3 = nn.BatchNorm2d(64)
        self.dropout = nn.Dropout(0.2)
        
        self.fc = nn.Linear(64*28*28, 105)

    def forward(self,x):
        x = self.pool1(F.relu(self.batch_norm1(self.conv1(x))))
        x = self.dropout(x)
        x = self.pool2(F.relu(self.batch_norm2(self.conv2(x))))
        x = self.dropout(x)
        x = self.pool3(F.relu(self.batch_norm3(self.conv3(x))))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [74]:
model = CustomCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNN().to(device)
model = model.to(device)
device

device(type='cpu')

In [75]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1)

In [76]:
from tqdm import tqdm
n_epochs = 15
img_size = 0
for epoch in tqdm(range(n_epochs)):
    model.train()
    running_loss = 0.0
    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)
        output = model(image)
        loss = criterion(output, label)
        running_loss += loss.item()*image.size(0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        img_size += 1 
    loss = running_loss/img_size
    tqdm.write(f'Epoch : {epoch+1}/{n_epochs} , Loss:{loss}')

  7%|▋         | 1/15 [02:35<36:11, 155.12s/it]

Epoch : 1/15 , Loss:1499.4844395617645


 13%|█▎        | 2/15 [05:18<34:39, 159.99s/it]

Epoch : 2/15 , Loss:6.947283700108528


 20%|██        | 3/15 [07:59<32:07, 160.63s/it]

...

Epoch : 13/15 , Loss:0.5827089574075106


 93%|█████████▎| 14/15 [36:17<02:33, 153.11s/it]

Epoch : 14/15 , Loss:0.5838970563241414


100%|██████████| 15/15 [38:50<00:00, 155.37s/it]

Epoch : 15/15 , Loss:0.5561925379615842


**Evaluation du modèle**

In [77]:
torch.save(model.state_dict(), "customcnn.pth")

In [78]:
model_loc = CustomCNN()
state_dict = torch.load('customcnn.pth')
model_loc.load_state_dict(state_dict)
model_loc.to(device)


/tmp/ipykernel_1523/1567053890.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('customcnn.pth')


CustomCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (batch_norm1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (batch_norm2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (batch_norm3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=50176, out_features=105, bias=True)
)

In [79]:
predictions = []
true_values = []
model_loc.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        output = model_loc(images)
        output = torch.argmax(output, dim=1)
        predictions.extend(output.cpu().numpy())
        true_values.extend(labels.cpu().numpy())

In [80]:
from sklearn.metrics import classification_report

report = classification_report(true_values, predictions)
print(report)

              precision    recall  f1-score   support

           0       0.91      0.92      0.91       262
           1       0.97      0.97      0.97       782

    accuracy                           0.95      1044
   macro avg       0.94      0.94      0.94      1044
weighted avg       0.96      0.95      0.96      1044



D'après les scores obtenus, on constate que le modèle est fiable lorsqu'il prédit des positifs et arrive à détecter correctement les vrais positifs.
Par la suite le modèle sera exposé à l'aide de FastAPI.